# Baseline — Standart Scaled Dot-Product Attention

Referans: GPT-2 Small'un kendi attention'ı. Diğer 9 yöntemin karşılaştırma noktası.

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/baseline_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
# Tüm hiperparametreler tek noktada. Sadece bu hücreyi düzenleyerek
# deneyi kontrol edebilirsin.

PARAMS = {
    # ─ Genel
    "method_name":   "baseline",            # bu notebook'a özgü, değiştirme
    "run_name":      "wt2_seed42",          # results/ altındaki klasör adı
    "seed":          42,
    "results_root":  "./results",           # LOKAL dizin (notebook yanında)

    # ─ Model
    "model_name":    "gpt2",                # gpt2 | gpt2-medium (RAM riskli)
    "tokenizer_name": "gpt2",

    # ─ Veri
    "dataset":       "wikitext2",           # wikitext2 | wikitext103 | ptb
    "max_seq_len":   512,
    "batch_size":    16,
    "num_workers":   2,

    # ─ Eğitim
    "learning_rate": 5e-5,
    "num_epochs":    3,
    "warmup_steps":  100,
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    # ─ Smoke test (None = tam veri)
    "limit_train_batches": 100,             # ilk denemede küçük tut
    "limit_eval_batches":  50,

    # ─ Log / Checkpoint
    "log_every_n_steps":         25,
    "save_checkpoints":          False,
    "checkpoint_every_n_steps":  500,
    "keep_last_n_checkpoints":   3,

    # ─ Method-specific (baseline'da yok)
    # Baseline GPT-2'nin kendi attention'ını olduğu gibi kullanır.
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method ───────────────────────────────────────────────────
# Baseline: GPT-2 Small'un standart attention'ını değiştirmiyoruz.
# Bu hücrede herhangi bir custom sınıf tanımı yok.
print("Baseline: HuggingFace GPT2 default attention kullanılacak.")

In [ ]:
# ─── HÜCRE 5: Build model & adapter ────────────────────────────────────
def build_model_and_adapter(params):
    from transformers import GPT2LMHeadModel
    model = GPT2LMHeadModel.from_pretrained(params["model_name"])
    return model, None

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL: {result['final_metrics']['test_ppl']:.4f}")
print(f"Final test BPC: {result['final_metrics']['test_bpc']:.4f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")